# Task 1: Coffee Sales Analysis

**Objective:** Transform raw transaction logs into actionable operational forecasts to optimize staffing and inventory.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# We will load the dataset, assuming it is named 'index.csv'
try:
    df = pd.read_csv('index.csv')
except:
    print('Dataset not found locally.')
    df = None


## Data Ingestion & Quality Assurance


In [ ]:
def ingest_retail_logs(path):
    dtypes = {
        'coffee_name': 'category',
        'money': 'float32',
        'cash_type': 'category'
    }
    try:
        df = pd.read_csv(path, dtype=dtypes)
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
        return df.dropna(subset=['datetime'])
    except:
        return None

df = ingest_retail_logs('index.csv')

def filter_anomalies(data, column):
    if data is None: return None
    z_scores = np.abs(stats.zscore(data[column]))
    clean_data = data[z_scores < 3]
    anomalies = len(data) - len(clean_data)
    print(f'Isolated {anomalies} anomalies.')
    zero_vals = clean_data[clean_data[column] == 0]
    if len(zero_vals) > 0:
        print(f'Found {len(zero_vals)} $0 transactions.')
        clean_data = clean_data[clean_data[column] > 0]
    return clean_data

df = filter_anomalies(df, 'money')


## Feature Engineering


In [ ]:
def synthesize_time_features(data):
    if data is None: return None
    data['hour'] = data['datetime'].dt.hour
    data['h_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['h_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    data['is_weekend'] = data['datetime'].dt.dayofweek.isin([5, 6]).astype(int)
    data['is_morning_rush'] = data['hour'].between(7, 10).astype(int)
    data['payday_proximity'] = data['datetime'].dt.day.isin([15, 30, 31]).astype(int)
    return data

df = synthesize_time_features(df)
if df is not None: print(df.head())


## Time-Series Aggregation & Modeling


In [ ]:
if df is not None:
    daily = df.groupby(df['datetime'].dt.date).agg({'money': 'sum', 'coffee_name': 'count'}).rename(columns={'coffee_name': 'sales'})
    for i in range(1, 8): daily[f'lag_{i}'] = daily['sales'].shift(i)
    daily['ema_7'] = daily['sales'].ewm(span=7).mean()
    daily.dropna(inplace=True)
    
    X = daily.drop(['sales', 'money'], axis=1)
    y = daily['sales']
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    
    rf_model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)
    rf_model.fit(X_train, y_train)
    
    predictions = rf_model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    residuals = y_test - predictions
    bias = residuals.mean()
    print(f'MAE: {mae:.2f} units, Bias: {bias:.2f}')
